# SAIM v3: paired LOL-v2 Real experiment

Use a Windows Python 3.14 environment with a CUDA-enabled PyTorch installation. In PowerShell, from `D:\retinexformer_improvement\myRetinexformer` on branch `v3`:

```powershell
py -3.14 -m pip install -r requirements.txt jupyter
py -3.14 -m unittest tests.test_saim -v
py -3.14 -m jupyter notebook SAIM_Experiment.ipynb
```

Set the paths and `RUN_STEPS` below. Run 100 first; then change `RUN_STEPS` to 1000 and 5000 for separate runs. Each run loads the same author checkpoint. Training and checkpoint selection use a seeded holdout from Train. Test is opened only in the final cell after training.


In [ ]:
from pathlib import Path
import torch
from saim_experiment import run_experiment, evaluate_final_test

WORKSPACE = Path(r"D:\retinexformer_improvement")
DATASET_ROOT = WORKSPACE / "LOLv2" / "Real_captured"
CHECKPOINT_PATH = WORKSPACE / "pretrained_weights" / "LOL_v2_real.pth"
RESULT_ROOT = WORKSPACE / "saim_experiment_results"
RUN_STEPS = 100  # choose 100, 1000, or 5000; each uses its own output folder
RESUME = False  # True resumes this run's last.pth exactly

assert DATASET_ROOT.is_dir(), f"Dataset missing: {DATASET_ROOT}"
assert CHECKPOINT_PATH.is_file(), f"Pretrained checkpoint missing: {CHECKPOINT_PATH}"
assert torch.cuda.is_available(), "CUDA PyTorch is required for the Windows GPU experiment"
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint:", CHECKPOINT_PATH)


## Train both arms

Both arms use the same pretrained weights, training examples, crop and augmentation at each optimizer update. Validation is full-resolution and never reads Test. At step 0, baseline and SAIM outputs are checked for equality. The run saves checkpoints, curves, tables, diagnostics, a manifest, and a progress log under `saim_experiment_results/updates_XXXX`. Set `RESUME=True` after interruption; keep the configuration and source checkpoint unchanged.


In [ ]:
run_dir = run_experiment(
    WORKSPACE,
    steps=RUN_STEPS,
    dataset_root=DATASET_ROOT,
    checkpoint_path=CHECKPOINT_PATH,
    result_root=RESULT_ROOT,
    resume=RESUME,
)
print("Results:", run_dir)


## Final held-out Test evaluation

Run this only after the selected experiment has completed. It loads each arm's best validation checkpoint, appends paired per-image Test results, and writes `final_test_summary.json`.


In [ ]:
summary = evaluate_final_test(run_dir, DATASET_ROOT)
print(summary)
